In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.metrics import mean_squared_error

In [17]:
# load the insurance data
data = pd.read_csv('insurance-1.csv')

print("Shape of dataset:", data.shape)
data.head()

Shape of dataset: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [18]:

print(data.isna().sum())

data = data.dropna()
print("Shape after removing missing rows:", data.shape)

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
Shape after removing missing rows: (1338, 7)


In [19]:
data = pd.get_dummies(data, drop_first=True)
data.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,False,True,False,False,True
1,18,33.770,1,1725.55230,True,False,False,True,False
2,28,33.000,3,4449.46200,True,False,False,True,False
3,33,22.705,0,21984.47061,True,False,True,False,False
4,32,28.880,0,3866.85520,True,False,True,False,False


In [20]:
# remove

numeric_data = data.select_dtypes(include=[np.number])

Q1 = numeric_data.quantile(0.25)
Q3 = numeric_data.quantile(0.75)
IQR = Q3 - Q1

data = data[~((numeric_data < (Q1 - 1.5 * IQR)) |
               (numeric_data > (Q3 + 1.5 * IQR))).any(axis=1)]
print("Shape after removing outliers:", data.shape)

Shape after removing outliers: (1193, 9)


In [21]:
X = data.drop('charges', axis=1)
y = data['charges']

In [22]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (954, 8)
Testing data: (239, 8)


In [24]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
mse_lr = mean_squared_error(y_test, y_pred_lr)

print("Linear Regression MSE:", mse_lr)

Linear Regression MSE: 19219976.00767454


In [25]:
for d in range(2,6):

    poly = PolynomialFeatures(degree=d)
    X_poly = poly.fit_transform(X_scaled)

    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_poly, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train_p, y_train_p)

    y_pred = model.predict(X_test_p)
    mse = mean_squared_error(y_test_p, y_pred)

    print("Polynomial Degree", d, "MSE:", mse)

Polynomial Degree 2 MSE: 17936496.932131156
Polynomial Degree 3 MSE: 20050475.114247262
Polynomial Degree 4 MSE: 57204871.082229406
Polynomial Degree 5 MSE: 15633004603.163588


In [26]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
mse_dt = mean_squared_error(y_test, y_pred_dt)
print("Decision Tree MSE:", mse_dt)

Decision Tree MSE: 43867917.86038991


In [27]:
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
print("Random Forest MSE:", mse_rf)

Random Forest MSE: 20664447.53093123


In [28]:
ada = AdaBoostRegressor(random_state=42)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)
mse_ada = mean_squared_error(y_test, y_pred_ada)

print("AdaBoost MSE:", mse_ada)

AdaBoost MSE: 34738139.6879432


In [29]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "AdaBoost"
    ],
    "MSE":[
        mse_lr,
        mse_dt,
        mse_rf,
        mse_ada
    ]
})

results

,Model,MSE
0,Linear Regression,1.921998e+07
1,Decision Tree,4.386792e+07
2,Random Forest,2.066445e+07
3,AdaBoost,3.473814e+07
